In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

input_text = "The capital of France is"
input_ids = tokenizer.encode(input_text, return_tensors="pt")

max_length = 20
print("生成过程 (启用 KV Cache):")

# 初始化 past_key_values
past_key_values = None
with torch.no_grad():
    for _ in range(max_length):
        # 前向计算，传入 past_key_values
        outputs = model(
            input_ids=input_ids[:, -1:] if past_key_values else input_ids,  # 仅输入最后一个token（如果已有缓存）
            past_key_values=past_key_values,  # 传入缓存的KV
            use_cache=True  # 显式启用缓存
        )
        
        # 更新 past_key_values
        past_key_values = outputs.past_key_values
        
        # 预测下一个token
        next_token_logits = outputs.logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        
        # 打印当前生成的token
        decoded_token = tokenizer.decode(next_token[0], skip_special_tokens=True)
        print(f"生成的token: {decoded_token!r}")

        # 终止条件
        if next_token == tokenizer.eos_token_id:
            break

        # 将新token添加到输入中
        input_ids = torch.cat([input_ids, next_token], dim=-1)

print("\n完整生成结果:")
print(tokenizer.decode(input_ids[0], skip_special_tokens=True))

In [2]:
n = 4 

nums = [1, 2, 3, 4]

post = []

for i in range(n-1, -1, -1):
    if not post:
        post.append(nums[i])
    else:
        post.append(nums[i]*post[-1])

ans = []
pre = 1
for i in range(n):
    post.pop()
    if post:
        ans.append(post[-1]*pre)
    else:
        ans.append(pre)
    pre *= nums[i]

print(*ans)
    

24 12 8 6


In [ ]:
x = 0

def dfs():
    global x
    x = 1

dfs()
print(x)

1


In [8]:
import math
from functools import cache
# coins = list(map(int, input().split()))
coins = [1, 2, 5]

amount = 11
ans = math.inf

@cache
def dfs(amount, n):
    global ans
    if amount==0:
        return n

    for coin in coins:
        if amount - coin >= 0:
            ans = min(dfs(amount-coin, n+1), ans)

    return math.inf

dfs(amount, 0)
print(ans)


3


In [ ]:
import torch
import torch.nn as nn
from transformers import ViTModel

class ViTAEDecoder(nn.Module):
    def __init__(self, latent_dim=768, patch_size=16, image_size=224):
        super().__init__()
        self.patch_size = patch_size
        self.image_size = image_size
        self.num_patches = (image_size // patch_size) ** 2
        
        # Transformer解码器
        decoder_layer = nn.TransformerDecoderLayer(d_model=latent_dim, nhead=8)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=4)
        
        # 重建头
        self.reconstruction_head = nn.Sequential(
            nn.Linear(latent_dim, patch_size * patch_size * 3),
            nn.Sigmoid()  # 输出0-1之间的像素值
        )

    def forward(self, latent_tokens):
        # latent_tokens: (batch_size, num_patches, latent_dim)
        decoded = self.decoder(latent_tokens, latent_tokens)
        patches = self.reconstruction_head(decoded)
        
        # 将patch序列重组为图像
        batch_size = patches.shape[0]
        patches = patches.view(batch_size, self.num_patches, 3, 
                             self.patch_size, self.patch_size)
        patches = patches.permute(0, 2, 1, 3, 4)
        patches = patches.contiguous().view(batch_size, 3, 
                                          self.image_size, self.image_size)
        return patches

In [ ]:

from einops.layers.torch import Rearrange

class ViTDecoder(nn.Module):
    def __init__(self, image_size: Union[Tuple[int, int], int], patch_size: Union[Tuple[int, int], int],
                 dim: int, depth: int, heads: int, mlp_dim: int, channels: int = 3, dim_head: int = 64) -> None:
        super().__init__()
        image_height, image_width = image_size if isinstance(image_size, tuple) \
                                    else (image_size, image_size)
        patch_height, patch_width = patch_size if isinstance(patch_size, tuple) \
                                    else (patch_size, patch_size)

        assert image_height % patch_height == 0 and image_width % patch_width == 0, 'Image dimensions must be divisible by the patch size.'
        de_pos_embedding = get_2d_sincos_pos_embed(dim, (image_height // patch_height, image_width // patch_width))

        self.num_patches = (image_height // patch_height) * (image_width // patch_width)
        self.patch_dim = channels * patch_height * patch_width

        self.transformer = Transformer(dim, depth, heads, dim_head, mlp_dim)
        self.de_pos_embedding = nn.Parameter(torch.from_numpy(de_pos_embedding).float().unsqueeze(0), requires_grad=False)
        self.to_pixel = nn.Sequential(
            Rearrange('b (h w) c -> b c h w', h=image_height // patch_height),
            nn.ConvTranspose2d(dim, channels, kernel_size=patch_size, stride=patch_size)
        )

        self.apply(init_weights)

    def forward(self, token: torch.FloatTensor) -> torch.FloatTensor:
        x = token + self.de_pos_embedding
        x = self.transformer(x)
        x = self.to_pixel(x)

        return x

    def get_last_layer(self) -> nn.Parameter:
        return self.to_pixel[-1].weight

In [3]:
import torch
batch_length = 64
subsequent_mask = (1 - torch.triu(torch.ones((1, batch_length, batch_length)), diagonal=1)).bool()
print(subsequent_mask)
print(subsequent_mask.shape)

tensor([[[ True, False, False,  ..., False, False, False],
         [ True,  True, False,  ..., False, False, False],
         [ True,  True,  True,  ..., False, False, False],
         ...,
         [ True,  True,  True,  ...,  True, False, False],
         [ True,  True,  True,  ...,  True,  True, False],
         [ True,  True,  True,  ...,  True,  True,  True]]])
torch.Size([1, 64, 64])


In [ ]:
mask = torch.ones((1, 1, batch_length)).bool()

In [ ]:
import torch
pixel_values = torch.random()
img, img_fused = torch.split(pixel_values, [3, 3], dim=1)

In [9]:
import torch
import torch.nn as nn
from torch.nn import Identity
import torch.nn.functional as F

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer(approximate='none')
        self.drop1 = nn.Dropout(drop)
        self.norm = Identity()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop2 = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.norm(x)
        x = self.fc2(x)
        x = self.drop2(x)
        return x

class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.q_norm = Identity()
        self.k_norm = Identity()
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        
        q = self.q_norm(q)
        k = self.k_norm(k)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.ls1 = Identity()
        self.drop_path1 = Identity()
        
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, drop=drop)
        self.ls2 = Identity()
        self.drop_path2 = Identity()

    def forward(self, x):
        x = x + self.drop_path1(self.ls1(self.attn(self.norm1(x))))
        x = x + self.drop_path2(self.ls2(self.mlp(self.norm2(x))))
        return x

class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=14, in_chans=3, embed_dim=1152):
        super().__init__()
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = Identity()

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x

class AttentionPoolLatent(nn.Module):
    def __init__(self, latent_num=256, embed_dim=1152, num_heads=8, mlp_ratio=4., drop=0.):
        super().__init__()
        self.q = nn.Linear(embed_dim, embed_dim)
        self.kv = nn.Linear(embed_dim, embed_dim * 2)
        self.q_norm = Identity()
        self.k_norm = Identity()
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.proj_drop = nn.Dropout(drop)
        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
        
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = Mlp(in_features=embed_dim, hidden_features=mlp_hidden_dim, drop=drop)

    def forward(self, x):
        B, N, C = x.shape
        
        # 修改：对所有 tokens 计算 query，而不是只取第一个 token
        q = self.q(x)  # (B, N, C) 而不是 (B, 1, C)
        q = self.q_norm(q)
        
        kv = self.kv(x).reshape(B, N, 2, C).permute(2, 0, 1, 3)
        k, v = kv.unbind(0)
        k = self.k_norm(k)
        
        # 计算注意力权重 (B, N, N)
        attn = (q @ k.transpose(-2, -1)) * (C ** -0.5)
        attn = attn.softmax(dim=-1)
        
        # 加权聚合 (B, N, C)
        x = (attn @ v)  # 不压缩维度
        x = self.proj(x)
        x = self.proj_drop(x)
        
        # 残差连接 + MLP
        x = x + self.mlp(self.norm(x))
        return x  # (B, N, C)

class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=14, in_chans=3, embed_dim=1152, depth=27, 
                 num_heads=16, mlp_ratio=4., qkv_bias=True, drop_rate=0., attn_drop_rate=0.):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, 
                                     in_chans=in_chans, embed_dim=embed_dim)
        self.pos_drop = nn.Dropout(p=drop_rate)
        self.patch_drop = Identity()
        self.norm_pre = Identity()
        
        self.blocks = nn.Sequential(*[
            Block(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, 
                  qkv_bias=qkv_bias, drop=drop_rate, attn_drop=attn_drop_rate)
            for _ in range(depth)])
        
        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
        self.attn_pool = AttentionPoolLatent(embed_dim=embed_dim, num_heads=num_heads, 
                                           mlp_ratio=mlp_ratio, drop=drop_rate)
        self.fc_norm = Identity()
        self.head_drop = nn.Dropout(p=drop_rate)
        self.head = Identity()

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.pos_drop(x)
        x = self.patch_drop(x)
        x = self.norm_pre(x)
        
        x = self.blocks(x)
        x = self.norm(x) # (B, 256, 1152)
        
        x = self.attn_pool(x) # (B, 1, 1152)
        x = self.fc_norm(x)
        x = self.head_drop(x)
        x = self.head(x)
        return x

class PrismaticVisionBackbone(nn.Module):
    def __init__(self, img_size=224, patch_size=14, in_chans=3, embed_dim=1152, depth=27, 
                 num_heads=16, mlp_ratio=4., qkv_bias=True, drop_rate=0., attn_drop_rate=0.):
        super().__init__()
        self.featurizer = VisionTransformer(
            img_size=img_size,
            patch_size=patch_size,
            in_chans=in_chans,
            embed_dim=embed_dim,
            depth=depth,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            qkv_bias=qkv_bias,
            drop_rate=drop_rate,
            attn_drop_rate=attn_drop_rate
        )

    def forward(self, x):
        return self.featurizer(x)

In [11]:


class DecoderBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, 
                            attn_drop=attn_drop, proj_drop=drop)
        self.ls1 = Identity()
        self.drop_path1 = Identity()
        
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, drop=drop)
        self.ls2 = Identity()
        self.drop_path2 = Identity()

    def forward(self, x):
        x = x + self.drop_path1(self.ls1(self.attn(self.norm1(x))))
        x = x + self.drop_path2(self.ls2(self.mlp(self.norm2(x))))
        return x

class PatchUnembed(nn.Module):
    def __init__(self, img_size=224, patch_size=14, in_chans=3, embed_dim=1152):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.ConvTranspose2d(
            embed_dim, in_chans, 
            kernel_size=patch_size, 
            stride=patch_size
        )
        self.norm = Identity()

    def forward(self, x):
        B, N, C = x.shape
        assert N == self.num_patches, "Input sequence length doesn't match expected patches"
        
        # (B, N, C) -> (B, C, H, W)
        x = x.transpose(1, 2).reshape(B, C, 
                                     self.img_size // self.patch_size, 
                                     self.img_size // self.patch_size)
        x = self.proj(x)
        x = self.norm(x)
        return x

class ViTDecoder(nn.Module):
    def __init__(self, img_size=224, patch_size=14, in_chans=3, 
                 embed_dim=1152, depth=27, num_heads=16, 
                 mlp_ratio=4., qkv_bias=True, drop_rate=0., attn_drop_rate=0.):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        
        # Transformer blocks
        self.blocks = nn.Sequential(*[
            DecoderBlock(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio,
                       qkv_bias=qkv_bias, drop=drop_rate, attn_drop=attn_drop_rate)
            for _ in range(depth)])
        
        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
        self.patch_unembed = PatchUnembed(
            img_size=img_size, patch_size=patch_size,
            in_chans=in_chans, embed_dim=embed_dim
        )

    def forward(self, x):
        # Input shape: (B, 256, 1152)
        x = self.blocks(x)
        x = self.norm(x)
        x = self.patch_unembed(x)  # (B, 3, 224, 224)
        return x

In [10]:
vitencoder = VisionTransformer()

encoder_params = sum(p.numel() for p in vitencoder.parameters())
print(f"encoder_params: {encoder_params}") # 447004800

x = torch.randn((2, 3, 224, 224))
y = vitencoder(x)
print(y.shape)


encoder_params: 447004800
torch.Size([2, 256, 1152])


In [12]:
decoder = ViTDecoder()
decoder_params = sum(p.numel() for p in decoder.parameters())
print(f"decoder_params: {decoder_params}") # 447004800
z = decoder(y)
print(z.shape)


decoder_params: 431065731
torch.Size([2, 3, 224, 224])


In [1]:
import torch
x = torch.full((2, 4), fill_value=True)

print(x)

tensor([[True, True, True, True],
        [True, True, True, True]])


In [2]:
y = (1 - torch.triu(torch.ones((1, 4, 4)), diagonal=1)).bool()

print(y)

tensor([[[ True, False, False, False],
         [ True,  True, False, False],
         [ True,  True,  True, False],
         [ True,  True,  True,  True]]])


In [7]:
def create_patch_causal_mask(seq_len, num_patches):
    """
    创建patch级别的因果掩码
    Args:
        seq_len: 序列长度 (T)
        num_patches: 每个位置的patch数量 (N)
    Returns:
        mask: [T*N, T*N] 的布尔掩码
    """
    # 创建块对角矩阵
    mask = torch.tril(torch.ones(seq_len, seq_len))  # [T, T] 标准下三角
    
    # 扩展为patch级别
    mask = mask.repeat_interleave(num_patches, dim=0)  # 行扩展
    mask = mask.repeat_interleave(num_patches, dim=1)  # 列扩展
    
    return mask.bool()

mask = create_patch_causal_mask(5, 2)

print(mask.shape)

torch.Size([10, 10])


In [14]:
def create_causal_mask(seq_len, device="cpu"):
    # 生成下三角矩阵（对角线及以下为1，其余为0）
    return torch.tril(torch.ones(seq_len, seq_len, device=device)).bool()

input_mask = torch.tensor([[1, 1, 1, 0], [1, 1, 1, 1]])
# 结合普通mask和因果mask
batch_size, seq_len = input_mask.shape
causal_mask = create_causal_mask(seq_len)
print(causal_mask.shape)
# 普通mask（来自Tokenizer）广播到与因果mask相同维度
expanded_mask = input_mask.unsqueeze(1).unsqueeze(2)  # [batch_size, 1, 1, seq_len]

# 合并两种mask（逻辑与操作）
combined_mask = expanded_mask & causal_mask

print(combined_mask.shape)

print(combined_mask)

torch.Size([4, 4])
torch.Size([2, 1, 4, 4])
tensor([[[[1, 0, 0, 0],
          [1, 1, 0, 0],
          [1, 1, 1, 0],
          [1, 1, 1, 0]]],


        [[[1, 0, 0, 0],
          [1, 1, 0, 0],
          [1, 1, 1, 0],
          [1, 1, 1, 1]]]])


In [ ]:
import timm
import torch
model_id = "vit_so400m_patch14_siglip_224"
img_size = 224
act_layer = None
featurizer = timm.create_model(
    model_id,
    pretrained=False,
    num_classes=0,
    img_size=img_size,
    act_layer=act_layer,
)

x = torch.randn((1, 3, 224, 224))

y1 = featurizer(x)

print(y1.shape)

torch.Size([1, 1152])


In [15]:
y2 = featurizer.forward_features(x)
print(y2.shape)

torch.Size([1, 256, 1152])


In [13]:
from typing import Any, Callable, ClassVar, Dict, List, Optional, Tuple, Union
from functools import partial

# === Utility Functions for Monkey-Patching ===
def unpack_tuple(fn: Callable[[Any], Tuple[Any]]) -> Callable[[Any], Any]:
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        result = fn(*args, **kwargs)
        return result[0] if isinstance(result, tuple) else result

    return wrapper

num_blocks = len(featurizer.blocks)
featurizer.forward = unpack_tuple(partial(featurizer.get_intermediate_layers, n={num_blocks - 2}))

In [16]:
y3 = featurizer(x)
print(y3.shape)

torch.Size([1, 256, 1152])


In [3]:
done = True
truncated = False
import numpy as np
x = np.logical_or(done, truncated)

x.any()

True

In [ ]:
import torch

x = torch.tensor([1.1, 2.2, 3.3])

y = x.cpu().numpy()

print(y)

[1.1 2.2 3.3]


In [9]:
num_envs = 1
obs_shape = (12, 224, 224)
max_length = 10
buffer = np.empty((max_length//num_envs, num_envs, *obs_shape), dtype=np.float16)
obs = np.random.random((12, 224, 224))

buffer[0] = obs

obs.shape
buffer[1].shape

(1, 12, 224, 224)

In [4]:
import torch
x = torch.randn((1, 512, 4096))
y = list(torch.split(x, [256] * 2, dim=1))
print(y)

[tensor([[[ 0.6227, -1.5574, -0.8308,  ...,  1.0035,  0.9483,  1.2307],
         [-0.3478, -1.2116, -2.1069,  ...,  0.6749,  0.3471,  1.3809],
         [-0.2050, -1.6806,  2.2286,  ...,  0.6311, -0.6504, -0.3460],
         ...,
         [-1.5353, -0.9017, -0.2635,  ...,  0.6281, -0.6473, -0.7141],
         [-1.0234,  0.6489,  1.4548,  ..., -0.1266, -0.8211,  0.5159],
         [-0.2230, -1.0679,  0.9356,  ..., -0.4109,  0.8149,  0.2768]]]), tensor([[[ 0.5756, -0.5030,  0.2336,  ...,  0.1043, -0.5028, -0.0380],
         [ 0.2480,  0.4763,  1.2864,  ..., -0.0553,  1.2722,  0.3823],
         [ 1.7069, -0.6335, -0.2623,  ...,  0.6878, -0.4198,  1.4881],
         ...,
         [-1.0791,  1.6220, -1.3669,  ..., -0.7115, -0.1776, -0.6611],
         [-1.3833,  0.9016, -0.7235,  ..., -1.0162, -0.6665,  1.3267],
         [-0.2383,  0.8623,  0.2949,  ...,  1.0780,  1.5608, -1.2092]]])]


In [6]:
all_images = []
x = torch.randn((1, 6, 224, 224))
all_images.append(x)
all_images.append(x)
all_images = torch.cat(all_images, dim=1)
print(all_images.shape)

torch.Size([1, 12, 224, 224])


In [3]:
import pickle
with open("../experiments/robot/libero/sample_libero_spatial_observation.pkl", "rb") as file:
    observation = pickle.load(file)

print(observation['full_image'].shape)
print(type(observation['full_image']))

(224, 224, 3)
<class 'numpy.ndarray'>


In [11]:
import numpy as np
from einops import rearrange
import torch
# 假设 obs1 和 obs2 的形状是 (H, W, C)
obs1 = np.random.rand(64, 64, 3)  # 示例数据 (H=64, W=64, C=3)
obs2 = np.random.rand(64, 64, 3)  # 示例数据 (H=64, W=64, C=3)

# 1. 调整轴顺序：(H, W, C) -> (C, H, W)
obs1_transposed = rearrange(torch.Tensor(obs1), "H W C -> C H W")/255.0  # 或 obs1.transpose(2, 0, 1)
obs2_transposed = rearrange(torch.Tensor(obs2), "H W C -> C H W")/255.0  # 或 obs2.transpose(2, 0, 1)

# 2. 合并：(C, H, W) + (C, H, W) -> (2*C, H, W)
full_obs = torch.cat([obs1_transposed, obs2_transposed], dim=0)

print("obs1 shape:", obs1.shape)              # (64, 64, 3)
print("obs1_transposed shape:", obs1_transposed.shape)  # (3, 64, 64)
print("full_obs shape:", full_obs.shape)       # (6, 64, 64)

obs1 shape: (64, 64, 3)
obs1_transposed shape: torch.Size([3, 64, 64])
full_obs shape: torch.Size([6, 64, 64])


In [10]:
print(obs1_transposed==full_obs[:3])

tensor([[[True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         ...,
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True]],

        [[True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         ...,
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True]],

        [[True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         ...,
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True],
         [True, True, True,  ..., True, True, True]]]

In [ ]:
import torch



torch.Size([4, 5, 6])
